In [1]:
import sys
from pathlib import Path

In [2]:
# in jupyter (lab / notebook), based on notebook path
module_path = str(Path.cwd().parents[0])

# # in standard python
# module_path = str(Path.cwd(__file__).parents[0] / "py")

if module_path not in sys.path:
    sys.path.append(module_path)

In [13]:
import data.donut_dataset as donut_dataset
import utils.helpers as helpers

from datasets import load_dataset
from torch.utils.data import DataLoader
from data.donut_dataset import DonutDataset
from models.donut_pytorch_lightning import DonutModelPLModule

In [4]:
image_path = "/Users/WilliamLiu/HeR_T_retaining/data/img_test"
max_length = 768

In [5]:
dataset = donut_dataset.data_loader(image_path)

print('Data Loading completes.')

this is the dataset DatasetDict({
    train: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 53
    })
    validation: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 30
    })
    test: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 30
    })
})
Data Loading completes.


In [6]:
processor, model = donut_dataset.model_loader(dataset, max_length, "naver-clova-ix/donut-base")

print('Processor and Model are loaded.')

/Users/WilliamLiu/miniforge3/envs/HeR-T/lib/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Processor and Model are loaded.


In [7]:
processor.image_processor.size = helpers.image_size(dataset)
processor.image_processor.do_align_long_axis = False

print("Processor's setting is completed.")

Processor's setting is completed.


In [8]:
train_dataset = DonutDataset(image_path, max_length=max_length,
                             split="train", task_start_token="<s_herbarium>", 
                             prompt_end_token="<s_herbarium>",
                             sort_json_key=False, # cord dataset is preprocessed -> no need for this
                             model=model, 
                             processor=processor,
                             )

val_dataset = DonutDataset(image_path, max_length=max_length,
                             split="validation", task_start_token="<s_herbarium>", 
                             prompt_end_token="<s_herbarium>",
                             sort_json_key=False, # cord dataset is preprocessed -> no need for this
                             model=model,
                             processor=processor,
                             )

print("DonutDataset is loaded.")

/Users/WilliamLiu/miniforge3/envs/HeR-T/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:890: UserWarning: Truncated File Read
  warnings.warn(str(msg))


DonutDataset is loaded.


In [9]:
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(['<s_herbarium>'])[0]

print("Model's setting is completed.")

Model's setting is completed.


In [10]:
train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=2)
val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)

print('Training and validation data loaders are created.')

Training and validation data loaders are created.


In [11]:
config = {
    'max_epochs': 20,
    'val_check_interval': 0.25,
    'check_val_every_n_epoch': 1,
    'gradient_clip_val': 1.0,
    'num_training_samples_per_epoch': 36760,
    'lr': 2.5e-5, # or 2e-5
    'weight_decay': 2e-5,
    'dropout_rate': 0.2,
    'train_batch_sizes': [8],
    'val_batch_sizes': [1],
    'num_nodes': 1,
    'warmup_steps': 2500,
    'result_path': "/kaggle/working/output/result",
    'verbose': True
}

In [14]:
model_lightning = DonutModelPLModule(config, processor, model)

print('PyTorch Lightning Model has been set up.')

PyTorch Lightning Model has been set up.
